# REBUILD CELLS



In [ ]:
%cd /content
!git clone https://github.com/yastrebksv/TennisProject.git
!pip install catboost yt-dlp scenedetect -q
%cd /content/TennisProject
!gdown 1XEYZ4myUN7QT-NeBYJI0xteLsvs-ZAOl -O model_ball.pt
!gdown 1f-Co64ehgq4uddcQm1aFBDtbnyZhQvgG -O model_court.pt
!gdown 1Eo5HDnAQE8y_FbOftKZ8pjiojwuy2BmJ -O model_bounce.cbm
!find /content/TennisProject -name "*.py" -exec sed -i 's/np\.Inf/np.inf/g' {} +

from google.colab import drive
drive.mount('/content/drive')

import torch, torchvision, sys
sys.path.append('/content/TennisProject')
from ball_detector import BallDetector
from court_detection_net import CourtDetectorNet
from bounce_detector import BounceDetector

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', "ON CPU — Runtime > Change runtime type > T4 GPU"

models = (
    BallDetector('/content/TennisProject/model_ball.pt', device),
    CourtDetectorNet('/content/TennisProject/model_court.pt', device),
    BounceDetector('/content/TennisProject/model_bounce.cbm'),
    torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval(),
)
print("Models loaded on", device)

# EXTRACTOR

In [ ]:
"""
=============================================================================
TENNIS BALL MACHINE - RALLY EXTRACTION v2
=============================================================================
Extracts rich per-shot features for training a shot-placement policy.

WHY v2: the v1 zone-only features had ZERO predictive signal (33% baseline =
majority class). Tactical shot selection depends on WHERE THE PLAYERS ARE,
which v1 never captured. v2 fixes that.

PER-SHOT FEATURES CAPTURED
--------------------------
Identity / context
  rally_id, shot_num, player_side (near/far), player_name, handedness

Where the ball came to the hitter  (the STATE)
  recv_x, recv_y        precise court coords of the bounce they're answering
  recv_zone             12-zone label (kept - cheap)
  recv_depth            DEFENSE/CONTROL/PRESSURE/ATTACK/FINISH  (tactical band)

Where everyone is standing at the moment of contact  (the KEY MISSING PIECE)
  hitter_x, hitter_y    hitter's court position
  opp_x, opp_y          opponent's court position
  opp_open_side         which side of the court is open ('left'/'right'/'center')
  opp_displacement      how far the opponent is from their ideal center position

The shot they played  (the ACTION / LABEL)
  stroke                FOREHAND / BACKHAND
  tgt_x, tgt_y          precise coords where THIS shot landed  (= next bounce)
  tgt_zone              12-zone label of the target
  risk                  0..1 geometric risk of that placement (edges = risky)
  to_open_court         did they hit into the open side? (bool)

Outcome
  is_winner, rally_ended, contact_frame, bounce_frame
=============================================================================
"""

import os, cv2, csv, json, time
import numpy as np
import torch


# =============================================================================
# COURT GEOMETRY  (reference-court coordinate space)
# =============================================================================
X_LDBL, X_LSGL, X_CTR, X_RSGL, X_RDBL = 286, 423, 832, 1242, 1379
Y_FAR_BASE, Y_FAR_SVC, Y_NET, Y_NEAR_SVC, Y_NEAR_BASE = 561, 1110, 1748, 2386, 2935

COURT_X, COURT_Y, MARGIN = (286, 1379), (561, 2935), 30

# singles court half-dimensions (used for risk normalisation)
SINGLES_HALF_W = (X_RSGL - X_LSGL) / 2          # ~410
HALF_LEN       = (Y_NET - Y_FAR_BASE) / 2       # ~594

ZONES = {
    "far_alley_left":      (X_LDBL, X_LSGL, Y_FAR_BASE, Y_NET),
    "far_alley_right":     (X_RSGL, X_RDBL, Y_FAR_BASE, Y_NET),
    "near_alley_left":     (X_LDBL, X_LSGL, Y_NET, Y_NEAR_BASE),
    "near_alley_right":    (X_RSGL, X_RDBL, Y_NET, Y_NEAR_BASE),
    "far_service_left":    (X_LSGL, X_CTR,  Y_FAR_SVC, Y_NET),
    "far_service_right":   (X_CTR,  X_RSGL, Y_FAR_SVC, Y_NET),
    "far_backcourt_left":  (X_LSGL, X_CTR,  Y_FAR_BASE, Y_FAR_SVC),
    "far_backcourt_right": (X_CTR,  X_RSGL, Y_FAR_BASE, Y_FAR_SVC),
    "near_service_left":   (X_LSGL, X_CTR,  Y_NET, Y_NEAR_SVC),
    "near_service_right":  (X_CTR,  X_RSGL, Y_NET, Y_NEAR_SVC),
    "near_backcourt_left": (X_LSGL, X_CTR,  Y_NEAR_SVC, Y_NEAR_BASE),
    "near_backcourt_right":(X_CTR,  X_RSGL, Y_NEAR_SVC, Y_NEAR_BASE),
}


def zone_of(coord):
    if coord is None: return "unknown"
    cx, cy = coord
    for name, (x0, x1, y0, y1) in ZONES.items():
        if x0 <= cx <= x1 and y0 <= cy <= y1:
            return name
    return "out_of_bounds"


def depth_band(y, side):
    """
    Tactical depth band (the Defense -> Finish scale).
    A ball landing DEEP in your court pins you back (DEFENSE).
    A ball landing SHORT lets you attack (FINISH).
    """
    if side == 'far':
        # far court: y 561 (baseline) .. 1748 (net).  small y = deep
        depth = (y - Y_FAR_BASE) / (Y_NET - Y_FAR_BASE)
    else:
        # near court: y 2935 (baseline) .. 1748 (net).  large y = deep
        depth = (Y_NEAR_BASE - y) / (Y_NEAR_BASE - Y_NET)
    depth = max(0.0, min(1.0, depth))
    if depth < 0.20: return 'DEFENSE'     # pinned to the baseline
    if depth < 0.40: return 'CONTROL'
    if depth < 0.60: return 'PRESSURE'
    if depth < 0.80: return 'ATTACK'
    return 'FINISH'                        # short ball - put it away


def shot_risk(x, y, side):
    """
    Geometric risk of a placement: 0 = dead centre (safe), 1 = painting a line.
    Risk is driven by the THINNEST margin (sideline or baseline), because that
    is the margin that will burn you.
    """
    d_side = min(x - X_LSGL, X_RSGL - x)          # margin to nearest sideline
    if side == 'far':
        d_deep, d_short = y - Y_FAR_BASE, Y_NET - y
    else:
        d_deep, d_short = Y_NEAR_BASE - y, y - Y_NET
    d_len = min(d_deep, d_short)                  # margin to baseline or net

    side_risk  = 1 - (d_side / SINGLES_HALF_W)
    depth_risk = 1 - (d_len  / HALF_LEN)
    return round(float(max(side_risk, depth_risk)), 3)


def open_court(opp_x):
    """Which side is open, given where the opponent is standing."""
    if opp_x is None: return 'unknown'
    if opp_x < X_CTR - 120: return 'right'    # they're left -> right is open
    if opp_x > X_CTR + 120: return 'left'
    return 'center'


# =============================================================================
# BALL / BOUNCE HELPERS
# =============================================================================
def interpolate_gaps(ball_track, max_gap=60):
    track = list(ball_track)
    det = [i for i in range(len(track)) if track[i][0] is not None]
    for k in range(len(det) - 1):
        i0, i1 = det[k], det[k+1]
        gap = i1 - i0 - 1
        if 1 <= gap <= max_gap:
            x0, y0 = track[i0]; x1, y1 = track[i1]
            for j in range(1, gap + 1):
                t = j / (gap + 1)
                track[i0 + j] = (x0 + (x1-x0)*t, y0 + (y1-y0)*t)
    return track


def to_court(f, ball_track, H):
    pos, mat = ball_track[f], H[f]
    if pos[0] is None or mat is None: return None
    pt = np.array(pos, dtype=np.float32).reshape(1, 1, 2)
    o = cv2.perspectiveTransform(pt, mat)
    return (float(o[0, 0, 0]), float(o[0, 0, 1]))


def player_to_court(box, H_frame):
    """Transform a player's FEET (bottom-centre of box) into court coords."""
    if box is None or H_frame is None: return (None, None)
    foot = np.array([[[(box[0] + box[2]) / 2, box[3]]]], dtype=np.float32)
    p = cv2.perspectiveTransform(foot, H_frame)[0, 0]
    return (round(float(p[0]), 1), round(float(p[1]), 1))


def on_court(p, margin=MARGIN):
    if p is None: return None
    x, y = p
    return (COURT_X[0]-margin <= x <= COURT_X[1]+margin and
            COURT_Y[0]-margin <= y <= COURT_Y[1]+margin)


def dir_change(f, ball_track, w=3):
    ys = [ball_track[i][1] if 0 <= i < len(ball_track) and ball_track[i][0] is not None else None
          for i in range(f-w, f+w+1)]
    before = [y for y in ys[:w] if y is not None]
    after  = [y for y in ys[w+1:] if y is not None]
    if not before or not after or ys[w] is None: return None
    return ys[w] >= max(before) - 2 and ys[w] >= max(after) - 2


def classify_bounces(bounces, ball_track, H, cluster_window=8):
    """IN/OUT/FALSE, then collapse consecutive same-side IN bounces (keep latest)."""
    classified = []
    for f in sorted(bounces):
        p = to_court(f, ball_track, H)
        classified.append((f, p, on_court(p), dir_change(f, ball_track)))

    results, i, n = [], 0, len(classified)
    while i < n:
        f, p, oc, dc = classified[i]
        if oc is True:
            j = i
            while (j+1 < n and classified[j+1][2] is True
                   and classified[j+1][0] - classified[j][0] <= cluster_window):
                j += 1
            results.append((classified[j][0], classified[j][1], 'IN'))
            i = j + 1
        elif oc is False:
            had = any(r[2] == 'IN' for r in results)
            results.append((f, p, 'FALSE' if (had and dc) else 'OUT'))
            i += 1
        else:
            i += 1

    collapsed, run, prev = [], [], None
    for e in results:
        if e[2] != 'IN':
            if run: collapsed.append(run[-1]); run = []
            prev = None
            collapsed.append(e)
            continue
        s = 'far' if e[1][1] < Y_NET else 'near'
        if s == prev:
            run.append(e)
        else:
            if run: collapsed.append(run[-1])
            run = [e]; prev = s
    if run: collapsed.append(run[-1])
    return collapsed


# =============================================================================
# PLAYER TRACKING  (SPARSE - only near bounce frames.  ~3x faster)
# =============================================================================
def track_players_sparse(frames, kps_court, det_model, device, bounce_frames, window=30):
    def detect(img, min_score=0.85):
        t = torch.from_numpy(img.transpose(2,0,1)/255).unsqueeze(0).float().to(device)
        with torch.no_grad():
            pred = det_model(t)[0]
        return [b.cpu().numpy() for b, l, s in
                zip(pred['boxes'], pred['labels'], pred['scores'])
                if l.item() == 1 and s.item() > min_score]

    def d2line(p, a, b):
        a, b, p = np.array(a, float), np.array(b, float), np.array(p, float)
        if np.allclose(a, b): return np.linalg.norm(p - a)
        return abs(float(np.cross(b - a, p - a))) / np.linalg.norm(b - a)

    needed = set()
    for bf in bounce_frames:
        needed.update(range(bf, min(bf + window, len(frames))))

    far_t  = [None] * len(frames)
    near_t = [None] * len(frames)
    for f in sorted(needed):
        kc = kps_court[f]
        if kc is None: continue
        def pt(i): return np.array(kc[i]).flatten()[:2]
        fa, fb, na, nb = pt(0), pt(1), pt(2), pt(3)
        ppl = detect(frames[f])
        if not ppl: continue
        fbest, nbest = 1e9, 1e9
        for b in ppl:
            foot = ((b[0] + b[2]) / 2, b[3])
            df, dn = d2line(foot, fa, fb), d2line(foot, na, nb)
            if df < fbest: fbest, far_t[f]  = df, b
            if dn < nbest: nbest, near_t[f] = dn, b
    return far_t, near_t


# =============================================================================
# MAIN EXTRACTOR
# =============================================================================
def extract_shots_v2(clip_path, rally_id, models, device,
                     near_handed='left', far_handed='right',
                     near_name='NADAL', far_name='DJOKOVIC',
                     look_ahead=25, contact_thresh=120):
    ball_detector, court_detector, bounce_detector, det_model = models

    cap = cv2.VideoCapture(clip_path)
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.resize(f, (1280, 720)))
    cap.release()
    if len(frames) < 30:
        return []

    ball_track = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps_court = court_detector.infer_model(frames)
    bounces = bounce_detector.predict([b[0] for b in ball_track],
                                      [b[1] for b in ball_track])
    classified = classify_bounces(bounces, ball_track, H)

    in_bounces = [(bf, c) for bf, c, v in classified if v == 'IN']
    if len(in_bounces) < 2:
        return []

    far_t, near_t = track_players_sparse(
        frames, kps_court, det_model, device,
        [bf for bf, _ in in_bounces], window=look_ahead + 5)

    rows = []
    for i in range(len(in_bounces) - 1):
        bf,   recv  = in_bounces[i]        # the bounce the hitter is answering
        nbf,  tgt   = in_bounces[i + 1]    # where their shot LANDED
        side = 'far' if recv[1] < Y_NET else 'near'

        # the target must be on the OTHER side - otherwise a bounce was missed
        tgt_side = 'far' if tgt[1] < Y_NET else 'near'
        if tgt_side == side:
            continue

        track   = far_t  if side == 'far' else near_t
        opp_trk = near_t if side == 'far' else far_t
        handed  = far_handed if side == 'far' else near_handed
        name    = far_name   if side == 'far' else near_name

        # find the contact frame: ball closest to the hitter after the bounce
        best = None
        for f in range(bf, min(bf + look_ahead, len(ball_track))):
            b, box = ball_track[f], track[f]
            if b[0] is None or box is None: continue
            cx, cy = (box[0]+box[2])/2, (box[1]+box[3])/2
            d = np.hypot(b[0]-cx, b[1]-cy)
            if best is None or d < best[1]:
                best = (f, d, b[0], cx)

        if best is None or best[1] > contact_thresh:
            rows.append(dict(
                rally_id=rally_id, shot_num=i+1, player_side=side,
                player_name=name, handedness=handed,
                recv_x=round(recv[0],1), recv_y=round(recv[1],1),
                recv_zone=zone_of(recv), recv_depth=depth_band(recv[1], side),
                hitter_x=None, hitter_y=None, opp_x=None, opp_y=None,
                opp_open_side='unknown', opp_displacement=None,
                stroke='NO_CONTACT',
                tgt_x=None, tgt_y=None, tgt_zone=None,
                risk=None, to_open_court=None,
                is_winner=True, contact_frame=None, bounce_frame=bf))
            break                                    # a winner ENDS the rally

        cf, d, bx, cx = best

        # positions at the moment of contact
        hx, hy = player_to_court(track[cf],   H[cf])
        ox, oy = player_to_court(opp_trk[cf], H[cf])

        # stroke: which side of the body was the ball on?
        screen = 'right' if bx > cx else 'left'
        body   = screen if side == 'near' else ('left' if screen == 'right' else 'right')
        stroke = 'FOREHAND' if body == handed else 'BACKHAND'

        open_side = open_court(ox)
        disp = round(abs(ox - X_CTR), 1) if ox is not None else None
        tgt_left = tgt[0] < X_CTR
        to_open = (open_side == 'left' and tgt_left) or (open_side == 'right' and not tgt_left)

        rows.append(dict(
            rally_id=rally_id, shot_num=i+1, player_side=side,
            player_name=name, handedness=handed,
            recv_x=round(recv[0],1), recv_y=round(recv[1],1),
            recv_zone=zone_of(recv), recv_depth=depth_band(recv[1], side),
            hitter_x=hx, hitter_y=hy, opp_x=ox, opp_y=oy,
            opp_open_side=open_side, opp_displacement=disp,
            stroke=stroke,
            tgt_x=round(tgt[0],1), tgt_y=round(tgt[1],1),
            tgt_zone=zone_of(tgt),
            risk=shot_risk(tgt[0], tgt[1], tgt_side),
            to_open_court=to_open,
            is_winner=False, contact_frame=cf, bounce_frame=bf))

    return rows


FIELDS = ['rally_id','shot_num','player_side','player_name','handedness',
          'recv_x','recv_y','recv_zone','recv_depth',
          'hitter_x','hitter_y','opp_x','opp_y','opp_open_side','opp_displacement',
          'stroke','tgt_x','tgt_y','tgt_zone','risk','to_open_court',
          'is_winner','contact_frame','bounce_frame']

# DOWNLOAD

In [ ]:
# 1. deno (if runtime reset since last download)
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ["PATH"] += ":/root/.deno/bin"

# 2. download → clean name → Drive immediately (survives reset)
!yt-dlp --cookies YOUR_COOKIES.txt --remote-components ejs:github \
  -f "bv*[height<=720]+ba/b[height<=720]/best" \
  -o "/content/drive/MyDrive/practice_clip.mp4" \
  "https://www.youtube.com/watch?v=C4Gl-T2dtss"

# 3. trim 30s + force 25fps + H.264 (avoids AV1 decode + OOM)
!ffmpeg -y -i /content/drive/MyDrive/practice_clip.mp4 -t 30 \
  -c:v libx264 -preset veryfast -r 25 -an /content/drive/MyDrive/practice_30s.mp4

In [ ]:
!ffmpeg -y -i /content/drive/MyDrive/utr_clip.mp4 -t 30 -c:v libx264 -preset veryfast -r 25 -an /content/drive/MyDrive/utr_30s.mp4

In [ ]:
!pip install -q "scenedetect==0.5.6.1"

In [ ]:
!ffmpeg -y -i /content/pov1.mp4 -vf "drawtext=text='%{n}':x=20:y=20:fontsize=48:fontcolor=yellow:box=1:boxcolor=black@0.6" -c:v libx264 -preset veryfast /content/pov_numbered.mp4

In [ ]:
import cv2, numpy as np

cap = cv2.VideoCapture('/content/pov1.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
h, w = frames[0].shape[:2]

NET_Y = int(h * 0.50)      # <-- tune this until the drawn line sits just below the net
LOWER = np.array([20,  40, 150])
UPPER = np.array([45, 255, 255])

out = cv2.VideoWriter('/content/pov_debug.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

for i, f in enumerate(frames):
    img = f.copy()
    roi = f[NET_Y:, :]
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    m = cv2.inRange(hsv, LOWER, UPPER)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    for c in cnts:
        a = cv2.contourArea(c)
        if a < 4 or a > 5000:
            continue
        (x, y), r = cv2.minEnclosingCircle(c)
        circ = a / (np.pi * r * r + 1e-6)
        if circ < 0.45:                 # ball is round; rejects glare streaks
            continue
        if best is None or a > best[2]:
            best = (x, y + NET_Y, a)

    # draw the ROI boundary so you can verify it sits below the net
    cv2.line(img, (0, NET_Y), (w, NET_Y), (255, 255, 0), 3)
    cv2.putText(img, "ROI starts here", (20, NET_Y - 15),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 3)

    if best:
        cv2.circle(img, (int(best[0]), int(best[1])), 20, (0, 0, 255), 3)
        status, col = f"HIT a={int(best[2])}", (0, 255, 0)
    else:
        status, col = "miss", (0, 0, 255)

    cv2.putText(img, f"f{i}  {status}", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0,0,0), 8)
    cv2.putText(img, f"f{i}  {status}", (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, col, 3)
    out.write(img)

out.release()
print("wrote /content/pov_debug.mp4")

In [ ]:
!ffmpeg -y -i "/content/IMG_1559 2.MOV" -c:v libx264 -preset veryfast -an /content/pov_native.mp4
!ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate,nb_frames -of default=noprint_wrappers=1 /content/pov_native.mp4

In [ ]:
!ffmpeg -y -i /content/pov_debug.mp4 -c:v libx264 -pix_fmt yuv420p /content/pov_debug_h264.mp4

In [ ]:
from google.colab import files
up = files.upload()   # select your screenshot
img_path = list(up.keys())[0]
print("uploaded:", img_path)

In [ ]:
path = clips[0]
cap = cv2.VideoCapture(path)
fps = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
print(path, len(frames), "frames at", fps)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pov_clips
!cp /content/TennisProject/*.MOV /content/drive/MyDrive/pov_clips/

In [ ]:
import cv2, numpy as np
cap = cv2.VideoCapture('/IMG_8585.MOV')     # original, no ffmpeg
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
print(len(frames), "frames at", cap.get(cv2.CAP_PROP_FPS))

In [ ]:
import cv2, numpy as np, os, shutil

# ---------- load original frames ----------
cap = cv2.VideoCapture('/content/bang.MOV')
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
h, w = frames[0].shape[:2]
print(f"{len(frames)} frames @ {fps:.2f}fps, {w}x{h}")

# ---------- params ----------
NET_Y = int(h * 0.50)          # tune until cyan line sits just below the net
LOWER = np.array([20,  40, 150])
UPPER = np.array([45, 255, 255])

# ---------- detect + annotate into PNGs (lossless) ----------
tmp = '/content/dbg_frames'
shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)

track = []
for i, f in enumerate(frames):
    img = f.copy()
    roi = f[NET_Y:, :]
    m = cv2.inRange(cv2.cvtColor(roi, cv2.COLOR_BGR2HSV), LOWER, UPPER)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    best = None
    for c in cnts:
        a = cv2.contourArea(c)
        if a < 4 or a > 5000: continue
        (x, y), r = cv2.minEnclosingCircle(c)
        if a / (np.pi*r*r + 1e-6) < 0.45: continue
        if best is None or a > best[2]: best = (x, y + NET_Y, a)
    track.append((best[0], best[1]) if best else (None, None))

    cv2.line(img, (0, NET_Y), (w, NET_Y), (255,255,0), 2)
    if best:
        cv2.circle(img, (int(best[0]), int(best[1])), 25, (0,0,255), 3)
        label, col = f"f{i}  HIT a={int(best[2])}", (0,255,0)
    else:
        label, col = f"f{i}  miss", (0,0,255)

    # frame number on the SIDE, large, with black backing so it's readable
    cv2.putText(img, label, (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0,0,0), 10)
    cv2.putText(img, label, (30, 80), cv2.FONT_HERSHEY_SIMPLEX, 2.0, col, 4)
    cv2.imwrite(f'{tmp}/{i:05d}.png', img)

hits = [i for i,p in enumerate(track) if p[0] is not None]
print(f"hits: {len(hits)}/{len(track)} ({100*len(hits)//len(track)}%)")
print("hit frames:", hits[:40])

# ---------- encode ONCE, near-lossless ----------
!ffmpeg -y -framerate {fps} -i {tmp}/%05d.png -c:v libx264 -crf 12 -preset slow -pix_fmt yuv420p /content/pov_debug_hq.mp4
print("wrote /content/pov_debug_hq.mp4")

In [ ]:
!ffmpeg -y -framerate 30 -i /content/dbg_frames/%05d.png \
  -vf "scale=in_range=full:out_range=limited" \
  -c:v libx264 -crf 12 -preset slow -pix_fmt yuv420p \
  -color_range tv -colorspace bt709 -color_primaries bt709 -color_trc bt709 \
  /content/pov_debug_fixed.mp4

In [ ]:
from google.colab import files
files.download('/content/pov_debug_fixed.mp4')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import glob, cv2, numpy as np
clips = sorted(glob.glob('/content/drive/MyDrive/pov_clips/*.MOV'))
for i, c in enumerate(clips): print(i, c)

In [ ]:
VIDEO = clips[10]                 # change index to pick a clip
cap = cv2.VideoCapture(VIDEO)
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
print(VIDEO, len(frames), "frames")

In [ ]:
import cv2, numpy as np, base64
from IPython.display import display, HTML
from google.colab import output

_, buf = cv2.imencode('.png', frames[0])
b64 = base64.b64encode(buf).decode()
roi_pts = []
def _rec(x, y):
    roi_pts.append((float(x), float(y))); print(f"{len(roi_pts)}: ({x:.0f},{y:.0f})")
output.register_callback('notebook.roi_click', _rec)

display(HTML(f'''
<p><b>Click 7 points, in order:</b>
1) net @ left singles, 2) net @ centre, 3) service hash @ centre,
4) service line @ left singles, 5) baseline @ left singles,
6) net @ left doubles, 7) baseline @ left doubles.</p>
<canvas id="rc" style="border:1px solid #999;cursor:crosshair;"></canvas>
<script>
const img=new Image();
img.onload=()=>{{const c=document.getElementById('rc'),s=Math.min(1,1100/img.width);
c.width=img.width*s;c.height=img.height*s;const x2=c.getContext('2d');
x2.drawImage(img,0,0,c.width,c.height);let n=0;
const pxpts=[];

const fixedLines=[[4,3],[6,2],[1,5],[6,7]];

function extendToEdge(p1,p2,w,h){{
  const dx=p2.x-p1.x, dy=p2.y-p1.y;
  let tMax=Infinity;
  if(dx>0) tMax=Math.min(tMax,(w-p1.x)/dx);
  if(dx<0) tMax=Math.min(tMax,(0-p1.x)/dx);
  if(dy>0) tMax=Math.min(tMax,(h-p1.y)/dy);
  if(dy<0) tMax=Math.min(tMax,(0-p1.y)/dy);
  return {{x:p1.x+dx*tMax, y:p1.y+dy*tMax}};
}}

function drawLine(a,b,color){{
  x2.strokeStyle=color;x2.lineWidth=2;
  x2.beginPath();x2.moveTo(a.x,a.y);x2.lineTo(b.x,b.y);x2.stroke();
}}

c.addEventListener('click',e=>{{if(n>=7)return;const r=c.getBoundingClientRect();
const X=(e.clientX-r.left)/s,Y=(e.clientY-r.top)/s;
const px={{x:e.clientX-r.left, y:e.clientY-r.top}};
pxpts.push(px);
x2.fillStyle='red';x2.beginPath();x2.arc(px.x,px.y,5,0,7);x2.fill();
x2.fillStyle='yellow';x2.font='18px sans-serif';x2.fillText(++n,px.x+8,px.y);

if(n===7){{
  for(const [ia,ib] of fixedLines){{
    drawLine(pxpts[ia-1], pxpts[ib-1], 'red');
  }}
  // baseline (7->5), extended past 5 to frame edge
  const p5=pxpts[4], p7=pxpts[6];
  const ext5=extendToEdge(p7, p5, c.width, c.height);
  drawLine(p7, ext5, 'red');

  // 2->3, extended past 3 to frame edge
  const p2=pxpts[1], p3=pxpts[2];
  const ext3=extendToEdge(p2, p3, c.width, c.height);
  drawLine(p2, ext3, 'red');

  // connector between the two edge-exit points, parallel to bottom edge
  const yEdge=Math.max(ext5.y, ext3.y);
  drawLine({{x:ext5.x, y:yEdge}}, {{x:ext3.x, y:yEdge}}, 'cyan');
}}

google.colab.kernel.invokeFunction('notebook.roi_click',[X,Y],{{}});}});}};
img.src='data:image/png;base64,{b64}';
</script>'''))
roi_pts.clear()

In [ ]:
img_pts = np.array(roi_pts, dtype=np.float32)
world_pts = np.array([
    [-13.5, 39.0],   # 1 net @ left singles
    [  0.0, 39.0],   # 2 net @ centre
    [  0.0, 18.0],   # 3 service hash   (21 ft from net = 18 from baseline)
    [-13.5, 18.0],   # 4 service line @ left singles
    [-13.5,  0.0],   # 5 baseline @ left singles
    [-18.0, 39.0],   # 6 net @ left doubles
    [-18.0,  0.0],   # 7 baseline @ left doubles
], dtype=np.float32)

H, _ = cv2.findHomography(img_pts, world_pts)
for ip, wp in zip(img_pts, world_pts):
    q = H @ np.array([ip[0], ip[1], 1.0])
    got = np.array([q[0]/q[2], q[1]/q[2]])
    print(f"{wp} -> {got.round(1)}  err {np.linalg.norm(got-wp):.2f} ft")

Hinv = np.linalg.inv(H)
def to_px(fx, fy):
    q = Hinv @ np.array([fx, fy, 1.0]); return np.array([q[0]/q[2], q[1]/q[2]])

In [ ]:
h, w = frames[0].shape[:2]
print(w, "x", h)

In [ ]:
edges = [((-18,0),(0,0)), ((0,0),(0,39)), ((0,39),(-18,39)), ((-18,39),(-18,0))]
pts = []
for (a, b) in edges:
    for t in np.linspace(0, 1, 60, endpoint=False):
        pts.append(to_px(a[0] + t*(b[0]-a[0]), a[1] + t*(b[1]-a[1])))
pts = np.array(pts)

def clip(sub, inside, inter):
    out = []
    for i in range(len(sub)):
        cur, prv = sub[i], sub[i-1]
        ci, pi = inside(cur), inside(prv)
        if ci:
            if not pi: out.append(inter(prv, cur))
            out.append(cur)
        elif pi:
            out.append(inter(prv, cur))
    return np.array(out) if len(out) else np.empty((0,2))

def cutter(axis, val, keep_greater):
    def inside(p): return p[axis] >= val if keep_greater else p[axis] <= val
    def inter(p, q):
        t = (val - p[axis]) / (q[axis] - p[axis] + 1e-9)
        return p + t*(q - p)
    return inside, inter

for axis, val, kg in [(0,0,True), (0,w-1,False), (1,0,True), (1,h-1,False)]:
    ins, itr = cutter(axis, val, kg)
    pts = clip(pts, ins, itr)

poly = pts.astype(np.int32)
court_mask = np.zeros((h, w), np.uint8)
cv2.fillPoly(court_mask, [poly], 255)
print("ROI covers", int(court_mask.sum()/255), "px")

vis = frames[0].copy()
cv2.polylines(vis, [poly], True, (0,255,0), 4)
from google.colab.patches import cv2_imshow
cv2_imshow(vis)

In [ ]:
# ROI polygon: 6 -> 2 -> ext3 -> ext5 -> 7 -> back to 6
# 6, 2, 7 use raw clicked pixel coords (roi_pts). ext3/ext5 are frame-edge
# intersections of the extended 2->3 and 5->7 lines (pixel-space only,
# no court-space meaning, so they can't come from to_px).

def extend_to_edge(p1, p2, w, h):
    p1 = np.array(p1, dtype=np.float64)
    p2 = np.array(p2, dtype=np.float64)
    d = p2 - p1
    t_max = np.inf
    if d[0] > 0: t_max = min(t_max, (w - p1[0]) / d[0])
    if d[0] < 0: t_max = min(t_max, (0 - p1[0]) / d[0])
    if d[1] > 0: t_max = min(t_max, (h - p1[1]) / d[1])
    if d[1] < 0: t_max = min(t_max, (0 - p1[1]) / d[1])
    return p1 + d * t_max

h, w = frames[0].shape[:2]

p1_, p2_, p3_, p5_, p6_, p7_ = (np.array(roi_pts[i], dtype=np.float64) for i in [0,1,2,4,5,6])

ext3 = extend_to_edge(p2_, p3_, w, h)   # ray 2->3, extended past 3 to frame edge
ext5 = extend_to_edge(p7_, p5_, w, h)   # ray 7->5, extended past 5 to frame edge

roi_poly = np.array([p6_, p2_, ext3, ext5, p7_], dtype=np.int32)

court_mask = np.zeros((h, w), np.uint8)
cv2.fillPoly(court_mask, [roi_poly], 255)
print("ROI covers", int(court_mask.sum() / 255), "px")

vis = frames[0].copy()
cv2.polylines(vis, [roi_poly], True, (0, 255, 0), 4)
from google.colab.patches import cv2_imshow
cv2_imshow(vis)

In [ ]:
# Overlay: green = ROI polygon (clicked/extended), blue = full court per homography
court_edges = [(-18,0),(0,0),(0,39),(-18,39),(-18,0)]
court_px = np.array([to_px(x, y) for x, y in court_edges], dtype=np.int32)

vis2 = frames[0].copy()
cv2.polylines(vis2, [roi_poly], True, (0, 255, 0), 4)   # ROI - green
cv2.polylines(vis2, [court_px], True, (255, 0, 0), 3)    # full court per H - blue
cv2_imshow(vis2)

In [ ]:
import cv2, numpy as np
from google.colab.patches import cv2_imshow

HALF_W, HALF_L  = 18.0, 39.0        # rulebook constants
BALL_R, HALF_LW = 0.11, 0.17        # ft: ball radius, half the widest line
MARGIN = BALL_R + HALF_LW

def court_mask(H, shape, side, margin=0.0):
    hh, ww = shape[:2]
    ys, xs = np.mgrid[0:hh, 0:ww]
    st = np.stack([xs.astype(np.float64),
                   ys.astype(np.float64),
                   np.ones_like(xs, dtype=np.float64)], axis=-1)
    wc = st @ H.T
    a, b = wc[..., 0] / wc[..., 2], wc[..., 1] / wc[..., 2]

    if side == 'left':
        x0, x1 = -HALF_W - margin, 0.0          # centre seam stays exact
    else:
        x0, x1 = 0.0, HALF_W + margin
    y0, y1 = -margin, HALF_L + margin

    return (((a >= x0) & (a <= x1) & (b >= y0) & (b <= y1)) * 255).astype(np.uint8)

In [ ]:
import cv2, numpy as np, base64, json
from IPython.display import display, HTML

_, buf = cv2.imencode('.png', frames[0])
b64 = base64.b64encode(buf).decode()
H_list = H.tolist()

display(HTML(f'''
<p><b>Click anywhere on the court</b> to see where the homography thinks it landed.</p>
<div style="display:flex; gap:20px; align-items:flex-start;">
  <canvas id="frameCanvas" style="border:1px solid #999; cursor:crosshair;"></canvas>
  <canvas id="courtCanvas" width="260" height="560" style="border:1px solid #999; background:#2d5a3d;"></canvas>
</div>
<script>
const H = {json.dumps(H_list)};
const HALF_W = 18.0;
const NET_Y = 39.0;          // our calibrated half: y=0 baseline .. y=39 net
const FULL_L = 78.0;         // full court: mirrored, net sits at the midpoint
const SINGLES_W = 13.5;
const SERVICE_Y = 18.0;      // service line, 18ft from (each) baseline
const CENTRE_MARK = 0.5;
const PAD = 20;

function applyH(px, py) {{
  const qx = H[0][0]*px + H[0][1]*py + H[0][2];
  const qy = H[1][0]*px + H[1][1]*py + H[1][2];
  const qz = H[2][0]*px + H[2][1]*py + H[2][2];
  return [qx/qz, qy/qz];
}}

const cc = document.getElementById('courtCanvas');
const cx = cc.getContext('2d');
const sx = (cc.width - 2*PAD) / (2*HALF_W);
const sy = (cc.height - 2*PAD) / FULL_L;

// fy here is in FULL-COURT space: 0 = near baseline, 39 = net, 78 = far baseline
function toCanvas(fx, fy) {{
  return {{ x: PAD + (fx + HALF_W) * sx, y: cc.height - PAD - fy * sy }};
}}

function line(a, b, width) {{
  cx.lineWidth = width || 2;
  const p = toCanvas(a[0], a[1]), q = toCanvas(b[0], b[1]);
  cx.beginPath(); cx.moveTo(p.x, p.y); cx.lineTo(q.x, q.y); cx.stroke();
}}

function drawCourt(markerX, markerY) {{
  cx.clearRect(0, 0, cc.width, cc.height);
  cx.strokeStyle = '#fff';

  // doubles sidelines, full length (both halves)
  line([-HALF_W, 0], [-HALF_W, FULL_L]);
  line([ HALF_W, 0], [ HALF_W, FULL_L]);

  // near baseline, far baseline (mirrored), net (heaviest, at the middle)
  line([-HALF_W, 0], [HALF_W, 0], 3);
  line([-HALF_W, FULL_L], [HALF_W, FULL_L], 3);
  line([-HALF_W, NET_Y], [HALF_W, NET_Y], 4);

  // singles sidelines, full length
  line([-SINGLES_W, 0], [-SINGLES_W, FULL_L]);
  line([ SINGLES_W, 0], [ SINGLES_W, FULL_L]);

  // near service line + mirrored far service line
  line([-SINGLES_W, SERVICE_Y], [SINGLES_W, SERVICE_Y]);
  line([-SINGLES_W, FULL_L-SERVICE_Y], [SINGLES_W, FULL_L-SERVICE_Y]);

  // centre service line: each half's service line up to the net (not through it)
  line([0, SERVICE_Y], [0, NET_Y]);
  line([0, FULL_L-SERVICE_Y], [0, NET_Y]);

  // centre marks at each baseline
  line([0, 0], [0, CENTRE_MARK], 2);
  line([0, FULL_L], [0, FULL_L-CENTRE_MARK], 2);

  if (markerX !== undefined) {{
    const m = toCanvas(markerX, markerY);   // markerY already in calibrated (near-half) space, matches this convention directly
    cx.fillStyle = 'red';
    cx.beginPath(); cx.arc(m.x, m.y, 6, 0, 7); cx.fill();
  }}
}}
drawCourt();

const img = new Image();
img.onload = () => {{
  const fc = document.getElementById('frameCanvas');
  const s = Math.min(1, 900/img.width);
  fc.width = img.width*s; fc.height = img.height*s;
  const fx2 = fc.getContext('2d');
  fx2.drawImage(img, 0, 0, fc.width, fc.height);

  fc.addEventListener('click', e => {{
    const r = fc.getBoundingClientRect();
    const X = (e.clientX-r.left)/s, Y = (e.clientY-r.top)/s;

    fx2.drawImage(img, 0, 0, fc.width, fc.height);
    fx2.fillStyle = 'red';
    fx2.beginPath(); fx2.arc(e.clientX-r.left, e.clientY-r.top, 6, 0, 7); fx2.fill();

    const [courtX, courtY] = applyH(X, Y);
    drawCourt(courtX, courtY);
  }});
}};
img.src = 'data:image/png;base64,{b64}';
</script>
'''))

In [ ]:
Hinv = np.linalg.inv(H)
def to_px(fx, fy):
    q = Hinv @ np.array([fx, fy, 1.0]); return np.array([q[0]/q[2], q[1]/q[2]])

edges = [(((-18,0),(0,0)),    (0,0,255),   "baseline"),
         (((0,0),(0,39)),     (0,255,255), "centre"),
         (((0,39),(-18,39)),  (255,0,0),   "net"),
         (((-18,39),(-18,0)), (255,0,255), "doubles sideline")]

vis = frames[0].copy()
h, w = vis.shape[:2]
for (a,b), col, name in edges:
    prev = None
    for t in np.linspace(0,1,300):
        p = to_px(a[0]+t*(b[0]-a[0]), a[1]+t*(b[1]-a[1]))
        if 0 <= p[0] < w and 0 <= p[1] < h:
            q = tuple(p.astype(int))
            if prev is not None: cv2.line(vis, prev, q, col, 4)
            prev = q
        else:
            prev = None
cv2_imshow(vis)

In [ ]:
for (px, py) in [(300, 900), (450, 1000), (600, 1050), (250, 850)]:
    q = H @ np.array([float(px), float(py), 1.0])
    fx, fy = q[0]/q[2], q[1]/q[2]
    print(f"px({px},{py}) -> court({fx:6.2f},{fy:6.2f}) ft")

In [ ]:
vis = frames[0].copy()
for a, b in [((-22,39),(0,39)), ((-18,39),(-18,0)), ((-13.5,39),(-13.5,0))]:
    p1, p2 = to_px(*a), to_px(*b)
    cv2.line(vis, tuple(p1.astype(int)), tuple(p2.astype(int)), (0,0,255), 3)
cv2.polylines(vis, [poly], True, (0,255,0), 3)
cv2_imshow(vis)

In [ ]:
vis = frames[0].copy()

for i, p in enumerate(roi_pts):
    p = np.array(p, dtype=int)
    cv2.circle(vis, tuple(p), 12, (0,255,255), -1)
    cv2.putText(vis, str(i+1), tuple(p + np.array([16,-16])),
                cv2.FONT_HERSHEY_SIMPLEX, 1.4, (0,255,255), 4)

for wp, col in [((-13.5,39),(0,0,255)), ((-18,39),(0,0,255)),
                ((-13.5,21),(255,0,255)), ((-18,21),(255,0,255))]:
    q = to_px(*wp).astype(int)
    cv2.circle(vis, tuple(q), 18, col, 4)
    print(f"{wp} -> ({q[0]},{q[1]})")

print("clicked 5:", np.array(roi_pts[4], dtype=int),
      "  clicked 7:", np.array(roi_pts[6], dtype=int))
cv2_imshow(vis)

In [ ]:
import cv2, numpy as np, base64
from IPython.display import display, HTML
from google.colab import output

# ---- CHANGE THIS: either a video+frame, or an image path ----
SOURCE = ('image', img_path, None) # (type, path, frame_num)
# SOURCE = ('image', '/content/drive/MyDrive/some_court.jpg', None)  # for a still image

kind, path, fnum = SOURCE
if kind == 'video':
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, fnum)
    ret, frame = cap.read(); cap.release()
else:
    frame = cv2.imread(path)
assert frame is not None, "couldn't load frame/image"

_, buf = cv2.imencode('.png', frame)
b64 = base64.b64encode(buf).decode()

clicked_points = []
def _record(x, y):
    clicked_points.append((float(x), float(y)))
    print(f"point {len(clicked_points)}: ({x:.0f}, {y:.0f})")
output.register_callback('notebook.record_click', _record)

html = f'''
<p><b>Click the 4 SINGLES-court corners</b> (any order).</p>
<canvas id="c" style="border:1px solid #999; cursor:crosshair;"></canvas>
<script>
const img = new Image();
img.onload = () => {{
  const cv=document.getElementById('c'), s=Math.min(1,1000/img.width);
  cv.width=img.width*s; cv.height=img.height*s;
  const ctx=cv.getContext('2d'); ctx.drawImage(img,0,0,cv.width,cv.height);
  let n=0;
  cv.addEventListener('click',(e)=>{{
    if(n>=4)return;
    const r=cv.getBoundingClientRect();
    const x=(e.clientX-r.left)/s, y=(e.clientY-r.top)/s;
    ctx.fillStyle='red'; ctx.beginPath();
    ctx.arc(e.clientX-r.left,e.clientY-r.top,5,0,7); ctx.fill();
    ctx.fillStyle='yellow'; ctx.font='16px sans-serif';
    ctx.fillText((n+1),e.clientX-r.left+8,e.clientY-r.top);
    n++;
    google.colab.kernel.invokeFunction('notebook.record_click',[x,y],{{}});
  }});
}};
img.src='data:image/png;base64,{b64}';
</script>'''
clicked_points.clear()
display(HTML(html))

In [ ]:
def draw_court(frame, pts):
    ic = np.array(pts[:4], dtype=np.float32)
    ic = ic[np.argsort(ic[:,1])]
    top = ic[:2][np.argsort(ic[:2,0])]; bot = ic[2:][np.argsort(ic[2:,0])]
    ic = np.array([top[0],top[1],bot[1],bot[0]], dtype=np.float32)  # TL,TR,BR,BL
    CL,CW = 78.0,27.0
    wc = np.array([[0,0],[CW,0],[CW,CL],[0,CL]], dtype=np.float32)
    Hinv = np.linalg.inv(cv2.findHomography(ic, wc)[0])
    def Ln(a,b,c,d): return [(a,b),(c,d)]
    lines = [Ln(0,0,CW,0),Ln(0,CL,CW,CL),Ln(0,0,0,CL),Ln(CW,0,CW,CL),
             Ln(0,39,CW,39),Ln(0,18,CW,18),Ln(0,60,CW,60),Ln(CW/2,18,CW/2,60)]
    def w2i(p):
        q=Hinv@np.array([p[0],p[1],1.0]); return (int(q[0]/q[2]),int(q[1]/q[2]))
    out = frame.copy()
    for a,b in lines: cv2.line(out, w2i(a), w2i(b), (0,0,255), 2)
    for c in ic: cv2.circle(out, tuple(c.astype(int)), 8, (0,255,255), -1)
    return out

from google.colab.patches import cv2_imshow
cv2_imshow(draw_court(frame, clicked_points))

In [ ]:
%cd /content/TennisProject
import cv2, torch
from ball_detector import BallDetector
from bounce_detector import BounceDetector

# --- load frames ---
cap = cv2.VideoCapture('/content/drive/MyDrive/practice_30s.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ret, f = cap.read()
    if not ret: break
    frames.append(f)
cap.release()
print(f"loaded {len(frames)} frames @ {fps}fps")

# --- ball detection ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bd = BallDetector('model_ball.pt', device)
ball_track = bd.infer_model(frames)

# --- bounce detection ---
x_ball = [p[0] for p in ball_track]
y_ball = [p[1] for p in ball_track]
bounce_det = BounceDetector('model_bounce.cbm')
bounce_frames = set(bounce_det.predict(x_ball, y_ball))
print(f"bounces at frames: {sorted(bounce_frames)}")

# --- draw detections onto frames ---
out_frames = []
trail = []  # recent ball positions for a fading trail
for i, f in enumerate(frames):
    img = f.copy()
    x, y = ball_track[i]
    if x is not None:
        trail.append((int(x), int(y)))
    trail = trail[-8:]  # keep last 8
    # draw ball trail
    for j, (tx, ty) in enumerate(trail):
        cv2.circle(img, (tx, ty), 3, (0, 255, 255), -1)
    # draw current ball bigger
    if x is not None:
        cv2.circle(img, (int(x), int(y)), 6, (0, 0, 255), 2)
    # mark a bounce: big flash + label, persists a few frames
    if any(bf in bounce_frames for bf in range(max(0,i-3), i+1)):
        if x is not None:
            cv2.circle(img, (int(x), int(y)), 18, (0, 255, 0), 3)
            cv2.putText(img, "BOUNCE", (int(x)+20, int(y)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)
    out_frames.append(img)

# --- write video ---
h, w = frames[0].shape[:2]
out_path = '/content/drive/MyDrive/detections_output.mp4'
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
for img in out_frames:
    writer.write(img)
writer.release()
print("wrote", out_path)

# SEGMENT + CUT

In [ ]:
from scenedetect import detect, ContentDetector
import json, os, time, subprocess

t0 = time.time()
scenes = detect('/content/match2.mp4', ContentDetector(threshold=27), show_progress=True)
segs = [(s.get_seconds(), e.get_seconds(), e.get_seconds()-s.get_seconds())
        for s, e in scenes]
segs = [s for s in segs if 3 <= s[2] <= 40]
print(f"{len(segs)} candidate rallies in {(time.time()-t0)/60:.0f} min")

with open('/content/drive/MyDrive/segments_match2.json','w') as f:
    json.dump(segs, f)

os.makedirs('/content/drive/MyDrive/rallies_match2', exist_ok=True)
for i, (s, e, d) in enumerate(segs):
    out = f'/content/drive/MyDrive/rallies_match2/rally_{i:04d}.mp4'
    if os.path.exists(out): continue
    subprocess.run(['ffmpeg','-y','-loglevel','error','-ss',str(s),
                    '-i','/content/match2.mp4','-t',str(d),'-c','copy',out],
                   check=False)
    if i % 100 == 0: print(f"  cut {i}/{len(segs)}")
print("clips ready")

# COURT FILTER

In [ ]:
import cv2, json, os
clip_dir = '/content/drive/MyDrive/rallies_match2'
clips = sorted(c for c in os.listdir(clip_dir) if c.endswith('.mp4'))

real = []
for i, c in enumerate(clips):
    cap = cv2.VideoCapture(os.path.join(clip_dir, c))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, n//2)
    ret, f = cap.read()
    cap.release()
    if not ret: continue
    _, kps = models[1].infer_model([cv2.resize(f, (1280,720))])
    if kps[0] is not None:
        real.append(c)
    if i % 100 == 0:
        print(f"  {i}/{len(clips)} → {len(real)} real")

print(f"\n{len(real)} real rallies of {len(clips)}")
with open('/content/drive/MyDrive/real_rallies_match2.json','w') as f:
    json.dump(real, f)

# EXTRACT


In [ ]:
import time, csv, json, random, os

clip_dir = '/content/drive/MyDrive/rallies_match2'
with open('/content/drive/MyDrive/real_rallies_match2.json') as f:
    real_rallies = sorted(json.load(f))

random.seed(42)
subset = sorted(random.sample(real_rallies, min(150, len(real_rallies))))
print(f"Processing {len(subset)} of {len(real_rallies)} clips")

out_csv = '/content/drive/MyDrive/shots_v2_match2.csv'
done = set()
if os.path.exists(out_csv):
    with open(out_csv) as f:
        done = {r['rally_id'] for r in csv.DictReader(f)}
    print(f"Resuming — {len(done)} done")

t0 = time.time()
with open(out_csv, 'a', newline='') as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if not done: w.writeheader()
    for i, clip in enumerate(subset):
        if clip in done: continue
        try:
            rows = extract_shots_v2(os.path.join(clip_dir, clip), clip, models, device,
                                    near_handed='right', far_handed='right',
                                    near_name='PLAYER', far_name='PLAYER')
            for r in rows: w.writerow(r)
            f.flush()
        except Exception as e:
            print(f"  {clip} FAILED: {e}")
        if i % 10 == 0 and i > 0:
            rate = (time.time()-t0)/i
            print(f"  {i}/{len(subset)} | ~{rate*(len(subset)-i)/60:.0f} min left")

print(f"\nDone in {(time.time()-t0)/60:.0f} min → {out_csv}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import time, os, json
clip_dir = '/content/drive/MyDrive/rallies_match2'
with open('/content/drive/MyDrive/real_rallies_match2.json') as f:
    real = sorted(json.load(f))

for clip in real[:3]:
    t0 = time.time()
    rows = extract_shots_v2(os.path.join(clip_dir, clip), clip, models, device,
                            near_handed='right', far_handed='right',
                            near_name='PLAYER', far_name='PLAYER')
    print(f"\n{clip}: {len(rows)} shots in {time.time()-t0:.0f}s")
    for r in rows:
        print(f"  {r['shot_num']}. {r['player_side']:4s} {r['stroke']:10s} "
              f"tgt({r['tgt_x']},{r['tgt_y']}) risk={r['risk']}")

In [ ]:
import cv2, numpy as np, os, json

def render_annotated(clip_path, out_path, models, device,
                     near_handed='right', far_handed='right'):
    ball_detector, court_detector, bounce_detector, det_model = models

    cap = cv2.VideoCapture(clip_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    frames = []
    while True:
        ret, f = cap.read()
        if not ret: break
        frames.append(cv2.resize(f, (1280,720)))
    cap.release()

    ball_track = interpolate_gaps(ball_detector.infer_model(frames), max_gap=60)
    H, kps = court_detector.infer_model(frames)
    bounces = bounce_detector.predict([b[0] for b in ball_track],
                                      [b[1] for b in ball_track])
    classified = classify_bounces(bounces, ball_track, H)

    # map each bounce frame -> its verdict + court coords
    bmap = {}
    for bf, coord, v in classified:
        bmap[bf] = (v, coord)
    raw = set(bounces)          # what the detector fired on, before classification

    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (1280,720))
    for f in range(len(frames)):
        img = frames[f].copy()

        # ball trail
        for j in range(8):
            k = f - j
            if k >= 0 and ball_track[k][0] is not None:
                cv2.circle(img, (int(ball_track[k][0]), int(ball_track[k][1])),
                           max(1, 5-j//2), (0,255,255), -1)

        # is the ball currently tracked?
        tracked = ball_track[f][0] is not None
        cv2.putText(img, "BALL: TRACKED" if tracked else "BALL: LOST",
                    (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                    (0,255,0) if tracked else (0,0,255), 2)
        cv2.putText(img, f"F{f}", (1180, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        # bounce markers - hold them visible for 12 frames so you can see them
        for bf in range(max(0, f-12), f+1):
            if bf in bmap and ball_track[bf][0] is not None:
                v, coord = bmap[bf]
                col = {'IN':(0,255,0), 'OUT':(0,0,255), 'FALSE':(128,128,128)}.get(v,(255,255,255))
                cv2.circle(img, (int(ball_track[bf][0]), int(ball_track[bf][1])), 16, col, 3)
                if bf == f:
                    side = 'far' if coord and coord[1] < Y_NET else 'near'
                    cv2.putText(img, f"BOUNCE {v} ({side})", (20, 690),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 3)
            elif bf in raw and bf not in bmap and ball_track[bf][0] is not None:
                # detector fired but classify_bounces threw it away
                cv2.circle(img, (int(ball_track[bf][0]), int(ball_track[bf][1])),
                           16, (255,0,255), 2)

        out.write(img)
    out.release()

    in_n = sum(1 for v,_ in bmap.values() if v=='IN')
    print(f"{os.path.basename(clip_path)}: {len(raw)} raw bounces -> "
          f"{len(bmap)} classified -> {in_n} IN -> {max(0,in_n-1)} shots")
    lost = sum(1 for b in ball_track if b[0] is None)
    print(f"   ball lost in {lost}/{len(frames)} frames ({100*lost/len(frames):.0f}%)")
    return out_path

In [ ]:
import csv
from collections import defaultdict

clip_dir = '/content/drive/MyDrive/rallies_match2'
with open('/content/drive/MyDrive/shots_v2_match2.csv') as f:
    rows = [r for r in csv.DictReader(f) if r['rally_id'] != 'rally_id']

per = defaultdict(int)
for r in rows: per[r['rally_id']] += 1

# a spread: the suspicious 1-shot ones, some middling, and the longest
one_shot = [k for k,v in per.items() if v == 1][:4]
mid      = [k for k,v in per.items() if 4 <= v <= 6][:3]
longest  = sorted(per, key=per.get, reverse=True)[:3]

os.makedirs('/content/annotated', exist_ok=True)
for clip in one_shot + mid + longest:
    print(f"\n--- {clip} (extractor found {per[clip]} shots) ---")
    render_annotated(os.path.join(clip_dir, clip),
                     f'/content/annotated/{clip}', models, device)